In [0]:
catalog_name='products'

In [0]:
silver_products=spark.table(
    f'{catalog_name}.silver.silver_products')
silver_reviews=spark.table(
    f'{catalog_name}.silver.silver_reviews')
silver_tags=spark.table(
    f'{catalog_name}.silver.silver_tags')

In [0]:
print(silver_products.columns)
from pyspark.sql.functions import avg
avg_price_for_category=silver_products.groupBy('category')\
    .agg(avg('price').alias('avg_price'))
display(avg_price_for_category)

['product_id', 'title', 'description', 'category', 'price', 'discountPercentage', 'rating', 'stock', 'brand', 'sku', 'weight', 'warrantyInformation', 'shippingInformation', 'availabilityStatus', 'returnPolicy', 'minimumOrderQuantity', 'thumbnail', 'dim_width', 'dim_height', 'dim_depth', 'created_at', 'updated_at', 'barcode', 'qr_code', 'ingested_at', 'data_source']


category,avg_price
furniture,1199.9899999999998
fragrances,83.99
beauty,13.39
groceries,6.043333333333334


In [0]:
top_rate_products=silver_products.orderBy(silver_products.rating.desc()).select(
    'product_id',
    'title',
    'category',
    'rating',
    'price'
)
display(top_rate_products.limit(10))

product_id,title,category,rating,price
30,Kiwi,groceries,4.93,2.49
14,Knoll Saarinen Executive Conference Chair,furniture,4.88,499.99
20,Cooking Oil,groceries,4.8,4.99
11,Annibale Colombo Bed,furniture,4.77,1899.99
3,Powder Canister,beauty,4.64,14.99
22,Dog Food,groceries,4.55,10.99
17,Beef Steak,groceries,4.47,12.99
6,Calvin Klein CK One,fragrances,4.37,49.99
4,Red Lipstick,beauty,4.36,12.99
5,Red Nail Polish,beauty,4.32,8.99


In [0]:
low_stock_products=silver_products.filter('stock <15')\
    .select(
        'product_id',
        'title',
        'stock',
        'price'
    )
display(low_stock_products)

product_id,title,stock,price
15,Wooden Bathroom Sink With Mirror,7,799.99
9,Dolce Shine Eau de,4,69.99
20,Cooking Oil,10,4.99
23,Eggs,9,2.99
26,Green Chili Pepper,3,0.99
16,Apple,8,1.99


In [0]:
from pyspark.sql.functions import col
inv_value=silver_products.withColumn('inventory_value',col('stock')*col('price'))

display(inv_value.select(
    'product_id',
    'title',
    'inventory_value'
))

product_id,title,inventory_value
12,Annibale Colombo Sofa,149999.4
13,Bedside Table African Cherry,19199.36
14,Knoll Saarinen Executive Conference Chair,12999.74
15,Wooden Bathroom Sink With Mirror,5599.93
4,Red Lipstick,1182.09
5,Red Nail Polish,710.21
6,Calvin Klein CK One,1449.71
7,Chanel Coco Noir Eau De,7539.42
8,Dior J'adore,8819.019999999999
9,Dolce Shine Eau de,279.96


In [0]:
from pyspark.sql.functions import avg
review_stat=silver_reviews.groupBy('product_id')\
    .agg(
        avg("review_rate").alias('avg_review_score')
    )
display(review_stat)

product_id,avg_review_score
1,4.0
2,3.3333333333333335
3,3.3333333333333335
4,4.666666666666667
5,2.6666666666666665
6,3.0
7,4.666666666666667
8,4.333333333333333
9,4.333333333333333
10,3.3333333333333335


In [0]:
from pyspark.sql.functions import count 
pop_tags=silver_tags.groupBy('tag')\
    .agg(count('*').alias('tag_count'))\
    .orderBy('tag_count',ascending=False)
display(pop_tags.limit(10))

tag,tag_count
beauty,5
fragrances,5
furniture,5
perfumes,5
vegetables,3
meat,2
fruits,2
pet supplies,2
mascara,1
lipstick,1


In [0]:
gold_table=silver_products.alias('p')\
    .join(
        review_stat.alias('r'),
        on='product_id',
        how='left'
    ).withColumn('inventory_value',col('price')*col('stock'))\
    .select(
        'product_id',
        'title',
        'category',
        'brand',
        'price',
        'stock',
        'rating',
        'avg_review_score',
        'inventory_value'
            )
display(gold_table.limit(1))

product_id,title,category,brand,price,stock,rating,avg_review_score,inventory_value
27,Honey Jar,groceries,null,6.99,34,3.97,2.3333333333333335,237.66


In [0]:
gold_table.write\
    .format('delta')\
    .mode('overwrite')\
    .saveAsTable(f'{catalog_name}.gold.gold_products')